### Dimension Date table creation

In [0]:
from pyspark.sql import functions as F

In [0]:
%run /Workspace/Agmarknet/setup/utilities

In [0]:
dbutils.widgets.text('catalog','agmarknet')
dbutils.widgets.text('data source','daily_prices')

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data source')

print(catalog)
print(data_source)

In [0]:
df_dim_date = spark.sql(f"select distinct Arrival_Date from {catalog}.{silver_schema}.{data_source}")
df_dim_date.show(10)

In [0]:
df_dim_date = (
    df_dim_date.withColumn("Date_Key", F.date_format("Arrival_Date", "yyyyMMdd").cast('int'))
    .withColumn("Day",F.dayofmonth("Arrival_Date"))
    .withColumn("Month",F.month("Arrival_date"))
    .withColumn("Month_name",F.date_format("Arrival_Date","MMMM"))
    .withColumn("Quarter",F.quarter("Arrival_Date"))
    .withColumn("Year",F.year("Arrival_Date"))
    .withColumn("Weekday",F.date_format("Arrival_Date",'EEEE'))
    .withColumn("IsWeekend",F.when(
        F.dayofweek("Arrival_Date").isin(1,7),"Yes"
        ).otherwise("No")
    ).withColumn("YearQuarter",F.concat(F.col("Year"),F.lit('-Q'),F.quarter("Arrival_date")))   
)

In [0]:
df_dim_date.show(10)

In [0]:
df_dim_date.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{gold_schema}.dim_date")